# RQ3 — Cosa succede se togli il giocatore più importante?

**Progetto:** Reti dei passaggi delle semifinaliste del Mondiale 2022
**Insegnamento:** Analisi e Visualizzazione delle Reti Complesse · Final project
**Lezioni di riferimento:** NS06 (centralità), NS07 (robustezza), NS08 (modelli nulli)
**Lettura:** Albert, Jeong & Barabási (2000), *Error and attack tolerance of complex networks*

---

## La domanda in breve

> *Se togliamo il giocatore più centrale di una squadra, quanto si rompe la sua rete dei passaggi? E questo danno è davvero peggio di quello che farebbe togliere un giocatore qualsiasi?*

La domanda si può leggere in due modi. Dal punto di vista della **rete**, è l'esperimento di attacco mirato della Lezione 7, applicato qui a reti piccole, pesate, dirette e con molti archi. Dal punto di vista del **calcio**, ci chiediamo quanto le quattro semifinaliste del 2022 dipendessero da un singolo giocatore per far girare il pallone.

Il notebook è diviso in cinque parti:

| § | Sezione | Cosa facciamo |
|---|---|---|
| 1 | **Punto di partenza** | Carichiamo le quattro reti e ci portiamo dietro il *rapporto top-1/top-2 della betweenness* visto nella RQ1. |
| 2 | **Come misurare il danno** | $S(q)$ della NS07 non funziona qui, quindi usiamo due metriche alternative pensate per reti dense. |
| 3 | **Togliere un giocatore** | Confronto fra togliere il pivot e togliere un giocatore a caso (n = 200), e calcolo dello z-score. |
| 4 | **Togliere più giocatori** | Rimuoviamo i top-3 uno dopo l'altro, ricalcolando la betweenness ogni volta. |
| 5 | **Mettendo tutto insieme** | Il rapporto top-1/top-2 della RQ1 predice davvero quale squadra è più fragile? |

Ogni sezione finisce con un piccolo riepilogo (**Cosa abbiamo imparato**), così da poter ricostruire facilmente il filo del discorso per il report finale.


In [ ]:
# -----------------------------------------------------------------------------
# Setup standalone — nessuna dipendenza da netsci_utils.
# Le reti dei passaggi vengono ricostruite da StatsBomb con la stessa pipeline
# usata in RQ1, garantendo così la perfetta compatibilità fra i due notebook.
# -----------------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

import os
import pickle
import random
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, Arc, FancyBboxPatch
from textwrap import fill
from scipy.stats import pearsonr, spearmanr

# Riproducibilità: stesso seed in tutto il notebook.
RANDOM_SEED = 42
def set_seeds(seed=RANDOM_SEED):
    np.random.seed(seed)
    random.seed(seed)
set_seeds()

# High-resolution inline rendering.
try:
    from IPython import get_ipython
    get_ipython().run_line_magic("config", "InlineBackend.figure_format = 'retina'")
except Exception:
    pass

plt.rcParams["figure.dpi"]     = 150
plt.rcParams["savefig.dpi"]    = 220
plt.rcParams["savefig.bbox"]   = "tight"
plt.rcParams["axes.grid"]      = False
plt.rcParams["font.size"]      = 11
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.edgecolor"]    = "#475569"
plt.rcParams["axes.labelcolor"]   = "#0F172A"
plt.rcParams["xtick.color"]       = "#475569"
plt.rcParams["ytick.color"]       = "#475569"

# Typography scale.
TEXT = {
    "fig_title":   16,
    "fig_subtitle": 11,
    "panel_title": 12,
    "annotation":   9.5,
    "direct_label": 9.5,
    "kpi_number":  20,
    "kpi_label":    9.2,
    "micro":        8.8,
}

# Figure-size scale.
FIG = {
    "focus":    (8.0, 6.0),
    "standard": (10.0, 6.0),
    "wide":     (12.5, 6.0),
    "flagship": (13.5, 12.5),
    "grid":     (12.5, 11.0),
}

# Colour system.
INK      = "#0F172A"
INK_SOFT = "#475569"
MUTED    = "#E2E8F0"
BG       = "#FAFBFC"

DV_PALETTE = {
    "blue":   "#4C72B0",
    "orange": "#DD8452",
    "green":  "#55A868",
    "red":    "#C44E52",
    "purple": "#8172B2",
    "gray":   "#7F8589",
}
ACCENT    = DV_PALETTE["orange"]
HIGHLIGHT = DV_PALETTE["red"]
PITCH_BG  = "#1e3a2d"
PITCH_LN  = "#e6efe6"

TEAM_COLORS = {
    "Argentina": "#75AADB",
    "France":    "#002395",
    "Croatia":   "#C44E52",
    "Morocco":   "#006233",
}

LINE_COLORS = {
    "GK":  "#FCBF49",
    "DEF": "#3A86FF",
    "MID": "#E63946",
    "ATT": "#06D6A0",
}


# -----------------------------------------------------------------------------
# Layout helpers.
# -----------------------------------------------------------------------------
def style_panel(ax, title=None, subtitle=None, *, title_color=INK):
    if title:
        ax.set_title(title, loc="left", pad=14 if subtitle else 8,
                     fontsize=TEXT["panel_title"], fontweight="semibold",
                     color=title_color)
    if subtitle:
        ax.text(0.0, 1.02, subtitle, transform=ax.transAxes,
                ha="left", va="bottom",
                fontsize=TEXT["annotation"], color=INK_SOFT,
                fontstyle="italic")
    return ax


def text_below_axes(ax, text, *, y=-0.16, mono=True, box=True):
    bbox = (dict(boxstyle="round,pad=0.30", fc="white", ec=MUTED, lw=0.8,
                 alpha=0.97) if box else None)
    ax.text(0.5, y, text, transform=ax.transAxes, ha="center", va="top",
            fontsize=TEXT["annotation"],
            family="monospace" if mono else None,
            bbox=bbox, clip_on=False, zorder=20)


def add_kpi_strip(ax, kpis):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_axis_off()
    n = len(kpis)
    for i, (num, lab) in enumerate(kpis):
        x = (i + 0.5) / n
        ax.text(x, 0.62, num, ha="center", va="center",
                fontsize=TEXT["kpi_number"], fontweight="semibold", color=INK)
        ax.text(x, 0.22, lab, ha="center", va="center",
                fontsize=TEXT["kpi_label"], color=INK_SOFT)
    ax.plot([0.02, 0.98], [-0.02, -0.02], color=MUTED, lw=0.9, clip_on=False)


def draw_pitch(ax, color=PITCH_BG, line=PITCH_LN, lw=1.0):
    ax.add_patch(Rectangle((0, 0), 120, 80, fc=color, ec=line, lw=lw + 0.5))
    ax.plot([60, 60], [0, 80], color=line, lw=lw)
    ax.add_patch(Circle((60, 40), 9.15, fill=False, ec=line, lw=lw))
    ax.plot(60, 40, "o", color=line, ms=2)
    ax.add_patch(Rectangle((0, 18), 18, 44, fill=False, ec=line, lw=lw))
    ax.add_patch(Rectangle((102, 18), 18, 44, fill=False, ec=line, lw=lw))
    ax.add_patch(Rectangle((0, 30), 6, 20, fill=False, ec=line, lw=lw))
    ax.add_patch(Rectangle((114, 30), 6, 20, fill=False, ec=line, lw=lw))
    ax.plot(12, 40, "o", color=line, ms=2); ax.plot(108, 40, "o", color=line, ms=2)
    ax.add_patch(Arc((12, 40), 18.3, 18.3, angle=0, theta1=-53, theta2=53,
                     color=line, lw=lw))
    ax.add_patch(Arc((108, 40), 18.3, 18.3, angle=0, theta1=127, theta2=233,
                     color=line, lw=lw))
    ax.set_xlim(-2, 122); ax.set_ylim(-2, 82)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)


# -----------------------------------------------------------------------------
# Dati + costanti.
# -----------------------------------------------------------------------------
COMPETITION_ID = 43    # FIFA World Cup
SEASON_ID      = 106   # edizione 2022
TEAMS          = ["Argentina", "France", "Croatia", "Morocco"]
MIN_MINUTES    = 30
N_RANDOM       = 200
PIVOT_METRIC   = "weighted betweenness"
CACHE_PATH     = "./wc2022_graphs.pkl"   # cache locale; cancellala se ricostruisci

ALIASES = {
    "Lionel Andrés Messi Cuccittini": "Messi",
    "Rodrigo Javier De Paul":          "De Paul",
    "Nicolás Hernán Otamendi":         "Otamendi",
    "Enzo Fernandez":                  "Fernandez",
    "Kylian Mbappé Lottin":            "Mbappé",
    "Aurélien Djani Tchouaméni":       "Tchouaméni",
    "Antoine Griezmann":               "Griezmann",
    "Randal Kolo Muani":               "Kolo Muani",
    "Achraf Hakimi Mouh":              "Hakimi",
    "Yahia Attiyat allah":             "Attiyat-Allah",
    "Luka Modrić":                     "Modrić",
    "Mateo Kovačić":                   "Kovačić",
    "Marcelo Brozović":                "Brozović",
    "Joško Gvardiol":                  "Gvardiol",
    "Sofyan Amrabat":                  "Amrabat",
}
def short(n):
    return ALIASES.get(n, n.split()[-1])


def _classify_line(position_name):
    """Mappa la posizione StatsBomb al reparto (GK/DEF/MID/ATT)."""
    if not isinstance(position_name, str):
        return "MID"
    p = position_name.lower()
    if "goalkeeper" in p:
        return "GK"
    if "back" in p:
        return "DEF"
    if "wing" in p and "wing back" not in p:
        return "ATT"
    if "forward" in p or "striker" in p or "centre-forward" in p:
        return "ATT"
    if "midfield" in p:
        return "MID"
    return "MID"


def build_aggregated_network(team_name, match_ids, min_minutes=MIN_MINUTES):
    """
    Costruisce la rete di passaggio aggregata di una squadra sull'intero
    torneo. Stessa identica logica di RQ1.

    Restituisce un nx.DiGraph con:
      - peso archi   = numero di passaggi completati
      - 'distance'   = 1 / peso (per cammini più corti pesati)
      - nodi con attributi 'x', 'y' (posizione media), 'line' (reparto),
        'minutes' (minuti giocati nel torneo).
    """
    from statsbombpy import sb

    all_passes = []
    player_total_minutes = {}
    player_position = {}

    for mid in match_ids:
        events = sb.events(match_id=mid)

        # Stima minuti dal Starting XI
        lineups = events[(events["type"] == "Starting XI") &
                         (events["team"] == team_name)]
        for _, row in lineups.iterrows():
            tactics = row.get("tactics")
            if isinstance(tactics, dict):
                for p in tactics.get("lineup", []):
                    name = p["player"]["name"]
                    player_total_minutes[name] = player_total_minutes.get(name, 0) + 90
                    pos = p.get("position", {})
                    if isinstance(pos, dict):
                        player_position.setdefault(name, pos.get("name", ""))

        # Correggi i minuti per le sostituzioni
        subs = events[(events["type"] == "Substitution") &
                      (events["team"] == team_name)]
        for _, row in subs.iterrows():
            passer = row.get("player")
            sub_info = row.get("substitution")
            if not isinstance(sub_info, dict):
                continue
            name_off = passer["name"] if isinstance(passer, dict) else passer
            name_on  = sub_info["replacement"]["name"]
            minute   = row.get("minute", 90)
            if name_off in player_total_minutes:
                player_total_minutes[name_off] -= (90 - minute)
            else:
                player_total_minutes[name_off] = minute
            player_total_minutes[name_on] = (
                player_total_minutes.get(name_on, 0) + (90 - minute)
            )

        # Solo passaggi completati della squadra
        passes = events[
            (events["type"] == "Pass") &
            (events["team"] == team_name) &
            (events["pass_outcome"].isna())
        ].copy()

        passes["x_start"] = passes["location"].apply(
            lambda l: l[0] if isinstance(l, list) else np.nan)
        passes["y_start"] = passes["location"].apply(
            lambda l: l[1] if isinstance(l, list) else np.nan)

        passes_clean = passes[["player", "pass_recipient",
                               "x_start", "y_start"]].dropna(
            subset=["player", "pass_recipient"])
        all_passes.append(passes_clean)

    df = pd.concat(all_passes, ignore_index=True)

    # Filtro soglia minuti
    eligible = {p for p, m in player_total_minutes.items() if m >= min_minutes}
    df = df[df["player"].isin(eligible) & df["pass_recipient"].isin(eligible)]

    # Costruisci grafo diretto pesato
    edge_counts = df.groupby(["player", "pass_recipient"]).size().reset_index(
        name="weight")

    G = nx.DiGraph()
    for _, row in edge_counts.iterrows():
        w = int(row["weight"])
        G.add_edge(row["player"], row["pass_recipient"],
                   weight=w, distance=1.0 / w)

    # Posizioni medie sul campo + reparto
    avg_pos = df.groupby("player").agg(x=("x_start", "mean"),
                                        y=("y_start", "mean")).reset_index()
    for _, row in avg_pos.iterrows():
        if row["player"] in G.nodes():
            G.nodes[row["player"]]["x"] = row["x"]
            G.nodes[row["player"]]["y"] = row["y"]
            G.nodes[row["player"]]["minutes"] = player_total_minutes.get(row["player"], 0)
            G.nodes[row["player"]]["line"] = _classify_line(
                player_position.get(row["player"], ""))
    return G


def load_wc2022_semifinalists(force_rebuild=False):
    """
    Carica le quattro reti aggregate del Mondiale 2022.
    - Se esiste un pickle di cache locale e force_rebuild=False, lo legge.
    - Altrimenti ricostruisce tutto da StatsBomb (~60-90 secondi) e salva il
      pickle per le esecuzioni successive.
    """
    if not force_rebuild and os.path.exists(CACHE_PATH):
        with open(CACHE_PATH, "rb") as f:
            graphs = pickle.load(f)
        print(f"✓ Reti caricate dalla cache {CACHE_PATH}")
        return graphs

    print("Costruzione delle reti da StatsBomb (può richiedere ~1 minuto)...")
    from statsbombpy import sb
    matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)

    graphs = {}
    for team in TEAMS:
        team_matches = matches[(matches["home_team"] == team) |
                               (matches["away_team"] == team)]
        match_ids = team_matches["match_id"].tolist()
        print(f"  {team}: {len(match_ids)} partite...", flush=True)
        G = build_aggregated_network(team, match_ids)
        graphs[team] = G
        print(f"    -> {G.number_of_nodes()} nodi, {G.number_of_edges()} archi, "
              f"{sum(d['weight'] for _,_,d in G.edges(data=True))} passaggi")

    with open(CACHE_PATH, "wb") as f:
        pickle.dump(graphs, f)
    print(f"✓ Reti salvate in cache {CACHE_PATH}")
    return graphs


# -----------------------------------------------------------------------------
# Funzioni di analisi (convenzione 1/peso per cammini pesati).
# -----------------------------------------------------------------------------
def add_distance(G):
    H = G.copy()
    for u, v, d in H.edges(data=True):
        d["distance"] = 1.0 / max(d["weight"], 1e-9)
    return H


def avg_path_length(G):
    if G.number_of_nodes() < 2:
        return float("nan")
    largest = max(nx.weakly_connected_components(G), key=len)
    H = add_distance(G.subgraph(largest).copy())
    lengths = dict(nx.all_pairs_dijkstra_path_length(H, weight="distance"))
    values = [d for u, t in lengths.items() for v, d in t.items() if u != v]
    return float(np.mean(values)) if values else float("nan")


def efficiency(G):
    if G.number_of_nodes() < 2:
        return float("nan")
    H = add_distance(G)
    lengths = dict(nx.all_pairs_dijkstra_path_length(H, weight="distance"))
    values = [1.0 / d for u, t in lengths.items() for v, d in t.items()
              if u != v and d > 0]
    return float(np.mean(values)) if values else float("nan")


def weighted_betweenness(G):
    return nx.betweenness_centrality(add_distance(G), weight="distance",
                                     normalized=True)


def relative_damage(pre, post, metric_name):
    if np.isnan(pre) or np.isnan(post) or pre <= 0:
        return float("nan")
    if metric_name == "avg_path_length":
        return (post - pre) / pre
    return (pre - post) / pre


---
## 1. Punto di partenza — quattro squadre, un numero a testa

Ognuna delle quattro semifinaliste ha la sua rete: un grafo diretto e pesato con 20–23 nodi (i giocatori scesi in campo per almeno 30 minuti) e 281–314 archi (le coppie che si sono scambiate almeno un passaggio durante il torneo).

Da questa sezione ci portiamo dietro un solo numero per squadra: il **rapporto top-1 / top-2 della weighted betweenness**. È la misura di dominanza del pivot che abbiamo introdotto nella RQ1, e ci dice quanto il giocatore più centrale stacca il secondo. Se il rapporto è 1, i primi due sono praticamente equivalenti; se è 3, il pivot è tre volte più centrale di chiunque altro.


In [ ]:
graphs = load_wc2022_semifinalists()

# Build the §1 summary table and the dominance score per team.
rows = []
for team in TEAMS:
    G = graphs[team]
    btw = weighted_betweenness(G)
    sorted_btw = sorted(btw.values(), reverse=True)
    top1_top2 = sorted_btw[0] / sorted_btw[1] if sorted_btw[1] > 0 else float("inf")
    pivot = max(btw, key=btw.get)
    rows.append({
        "team": team,
        "n":    G.number_of_nodes(),
        "edges":    G.number_of_edges(),
        "total passes":  sum(d["weight"] for _, _, d in G.edges(data=True)),
        "density":      nx.density(G),
        "reciprocity":  nx.reciprocity(G),
        "pivot":        short(pivot),
        "top1/top2":    top1_top2,
    })
setup_df = pd.DataFrame(rows)
print(setup_df.round(2).to_string(index=False))


In [ ]:
# A visual summary of the four-team setup: a KPI strip across the top,
# followed by per-team mini-cards with the pivot identified.
fig = plt.figure(figsize=FIG["wide"], facecolor=BG)
gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[1.0, 2.6],
                      left=0.05, right=0.97, top=0.93, bottom=0.06,
                      hspace=0.35)

# --- Title and subtitle directly on the figure
fig.text(0.05, 0.985, "Quattro squadre, quattro reti dei passaggi — una fotografia strutturale",
         ha="left", va="top", fontsize=TEXT["fig_title"],
         fontweight="semibold", color=INK)
fig.text(0.05, 0.955,
         "Reti aggregate per squadra-torneo delle semifinaliste del Mondiale 2022 FIFA. "
         "Rapporto top-1 / top-2 = il divario fra il giocatore più centrale e il secondo più centrale.",
         ha="left", va="top", fontsize=TEXT["fig_subtitle"], color=INK_SOFT)

# --- KPI strip: aggregate totals across the four teams
ax_kpi = fig.add_subplot(gs[0])
total_passes = int(setup_df["total passes"].sum())
total_players = int(setup_df["n"].sum())
mean_density = setup_df["density"].mean()
mean_reciprocity = setup_df["reciprocity"].mean()
add_kpi_strip(ax_kpi, [
    (f"{total_passes:,}",         "passaggi completati aggregati"),
    (f"{total_players}",          "giocatori con ≥ 30′ giocati"),
    (f"{mean_density:.2f}",       "densità media delle reti"),
    (f"{mean_reciprocity:.2f}",   "reciprocità media (passaggio e ritorno)"),
])

# --- Per-team mini-cards
ax_cards = fig.add_subplot(gs[1]); ax_cards.set_axis_off()
ax_cards.set_xlim(0, 1); ax_cards.set_ylim(0, 1)
for i, row in setup_df.iterrows():
    x0 = 0.025 + i * 0.245; x1 = x0 + 0.22
    team = row["team"]; color = TEAM_COLORS[team]
    # Left coloured stripe
    ax_cards.add_patch(Rectangle((x0, 0.10), 0.012, 0.80,
                                 fc=color, ec="none", clip_on=False))
    # Card body
    ax_cards.add_patch(FancyBboxPatch((x0 + 0.015, 0.10), 0.205, 0.80,
        boxstyle="round,pad=0.005,rounding_size=0.012",
        fc="white", ec=MUTED, lw=0.8, clip_on=False))
    # Team name
    team_it = {"Argentina":"Argentina","France":"Francia","Croatia":"Croazia","Morocco":"Marocco"}[team]
    ax_cards.text(x0 + 0.028, 0.82, team_it, ha="left", va="center",
                  fontsize=13, fontweight="semibold", color=INK)
    # Pivot
    ax_cards.text(x0 + 0.028, 0.72, f"pivot · {row['pivot']}", ha="left",
                  va="center", fontsize=10.5, color=INK_SOFT, fontstyle="italic")
    # Top1/top2 ratio - the highlighted number
    ax_cards.text(x0 + 0.028, 0.50, f"{row['top1/top2']:.2f}", ha="left",
                  va="center", fontsize=22, fontweight="semibold", color=color)
    ax_cards.text(x0 + 0.028, 0.36, "rapporto top-1 / top-2 della betweenness",
                  ha="left", va="center", fontsize=8.8, color=INK_SOFT)
    # Three smaller stats
    ax_cards.text(x0 + 0.028, 0.22,
                  f"{row['n']} giocatori · {row['edges']} archi · {int(row['total passes']):,} passaggi",
                  ha="left", va="center", fontsize=8.5, color=INK_SOFT)
    ax_cards.text(x0 + 0.028, 0.15,
                  f"densità {row['density']:.2f}   ·   reciprocità {row['reciprocity']:.2f}",
                  ha="left", va="center", fontsize=8.5, color=INK_SOFT,
                  family="monospace")
plt.show()


**Cosa abbiamo imparato.**
- Tutte e quattro le reti sono **dense** (densità 0.63–0.85) e **molto reciproche** (0.88–0.93): in pratica, grafi quasi completi in cui quasi tutti i passaggi sono ricambiati.
- Il **Marocco** è quello fuori dal coro per volume: appena 2 124 passaggi totali contro i 3 100–3 800 delle altre tre. Coerente con uno stile di gioco meno orientato al possesso. La **Francia** invece ha la rotazione più ampia (22 giocatori sopra i 30′), il che ne abbassa meccanicamente la densità (0.63).
- Il **rapporto top-1 / top-2** varia di un ordine di grandezza fra le quattro squadre: da **1.04** (Croazia — Gvardiol e Modrić praticamente pari merito) a **3.56** (Francia — Tchouaméni nettamente sopra tutti).
- *Ipotesi che verifichiamo nella §5:* questo rapporto dovrebbe predire quanto ogni squadra soffre quando le togliamo il pivot.


---
## 2. Come misurare il danno — perché $S(q)$ qui non basta

La metrica di robustezza vista nella NS07,

$$ S(q) = \frac{\text{dimensione della LCC dopo aver rimosso una frazione } q}{N} $$

non funziona bene sulle nostre reti. Sono troppo dense (densità 0.57–0.75): **togliere un solo nodo non spezza mai la componente connessa principale** (lo abbiamo verificato su più di 800 rimozioni di prova — vedi §7 della proposta). Ci servono allora metriche che catturino il *peggioramento della rete prima ancora che si rompa*.

Ne usiamo due:

- **Lunghezza media dei cammini** sulla LCC, usando `1/peso` come distanza, così che passaggi frequenti corrispondano a percorsi brevi.
- **Efficienza** $E = \frac{1}{n(n-1)}\sum_{i \neq j} \frac{1}{d_{ij}}$ — la media degli inversi delle distanze fra tutte le coppie raggiungibili (Latora & Marchiori 2001). Più è bassa, più lentamente la palla circola nella rete.

In entrambi i casi riporteremo il **danno relativo**, scegliendo il segno in modo che *un valore positivo significhi sempre "peggio di prima"*.


In [ ]:
baseline_rows = []
for team in TEAMS:
    G = graphs[team]
    baseline_rows.append({
        "team":             team,
        "avg path length":  avg_path_length(G),
        "efficiency":       efficiency(G),
    })
baseline_df = pd.DataFrame(baseline_rows).set_index("team")
print(baseline_df.round(3).to_string())


**Cosa abbiamo imparato.**
- L'**Argentina** ha i cammini più corti e l'efficienza più alta: una rete di possesso fitta e ben collegata.
- Il **Marocco** ha i cammini più lunghi e l'efficienza più bassa — in media servono più passaggi per attraversare la rete, sempre coerente con uno stile meno basato sul possesso.
- Questi quattro valori (per due metriche) sono il *baseline* con cui confronteremo nella §3 il danno fatto dalla rimozione di un giocatore.


---
## 3. Togliere un giocatore — pivot vs giocatore a caso

Per ogni squadra ripetiamo l'esperimento della Lezione 7, in versione mini:

| | Cosa togliamo | Quante volte |
|---|---|---|
| **Mirato** | il giocatore top-1 per weighted betweenness (il pivot) | 1 |
| **Casuale** | un giocatore di movimento a caso (escluso il portiere e il pivot) | $N = 200$ |

Per ogni combinazione squadra × metrica, trasformiamo il danno mirato in **z-score** rispetto alla distribuzione delle 200 rimozioni casuali. Il p-value a una coda ci dice quante delle rimozioni casuali raggiungono o superano il danno fatto dal togliere il pivot.


In [ ]:
metrics = {"avg_path_length": avg_path_length, "efficiency": efficiency}

rows = []
for team in TEAMS:
    G = graphs[team]
    btw = weighted_betweenness(G); pivot = max(btw, key=btw.get)
    pre = {m: f(G) for m, f in metrics.items()}

    # Targeted
    Gt = G.copy(); Gt.remove_node(pivot)
    post = {m: f(Gt) for m, f in metrics.items()}
    for m in metrics:
        rows.append({"team": team, "mode": "targeted", "realisation": 0,
                     "metric": m,
                     "damage": relative_damage(pre[m], post[m], m)})

    # Random null
    rng = np.random.default_rng(RANDOM_SEED)
    outfield = [n for n in G.nodes() if G.nodes[n].get("line") != "GK" and n != pivot]
    for r in range(N_RANDOM):
        n_remove = rng.choice(outfield)
        Gr = G.copy(); Gr.remove_node(n_remove)
        post = {m: f(Gr) for m, f in metrics.items()}
        for m in metrics:
            rows.append({"team": team, "mode": "random", "realisation": r,
                         "metric": m,
                         "damage": relative_damage(pre[m], post[m], m)})

df = pd.DataFrame(rows)

# Summary table.
summary = []
for team in TEAMS:
    for m in metrics:
        sub = df[(df.team == team) & (df.metric == m)]
        t = sub[sub["mode"] == "targeted"]["damage"].iloc[0]
        rvs = sub[sub["mode"] == "random"]["damage"].dropna().values
        rmean, rstd = rvs.mean(), rvs.std(ddof=1)
        z = (t - rmean) / rstd if rstd > 0 else float("nan")
        p = float(np.mean(rvs >= t))
        summary.append({"team": team, "metric": m,
                        "targeted": round(t, 4),
                        "random mean": round(rmean, 4),
                        "random std":  round(rstd, 4),
                        "z": round(z, 2), "p": round(p, 3)})
summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))


In [ ]:
# Hero figure for §3: KPI band + 4×2 grid of histograms with targeted overlay.
_team_it_map = {"Argentina":"Argentina","France":"Francia","Croatia":"Croazia","Morocco":"Marocco"}
fig = plt.figure(figsize=FIG["grid"], facecolor=BG)
gs = fig.add_gridspec(nrows=5, ncols=2, height_ratios=[0.85, 1, 1, 1, 1],
                      left=0.07, right=0.97, top=0.93, bottom=0.05,
                      hspace=0.50, wspace=0.18)

fig.text(0.07, 0.985, "Rimuovere il pivot vs rimuovere un giocatore di movimento a caso",
         ha="left", va="top", fontsize=TEXT["fig_title"], fontweight="semibold",
         color=INK)
fig.text(0.07, 0.957,
         "Per ogni squadra e ogni metrica, il danno della rimozione mirata (linea rossa) è confrontato "
         "con la distribuzione dei danni prodotti da 200 rimozioni casuali di giocatori di movimento.",
         ha="left", va="top", fontsize=TEXT["fig_subtitle"], color=INK_SOFT)

# KPI strip across the top
ax_kpi = fig.add_subplot(gs[0, :])
ranked = summary_df.groupby("team")["z"].mean().sort_values(ascending=False)
most_fragile, least_fragile = ranked.index[0], ranked.index[-1]
max_z = summary_df["z"].max()
mean_z = summary_df["z"].mean()
add_kpi_strip(ax_kpi, [
    (f"{N_RANDOM}",                     "realizzazioni casuali per squadra"),
    (f"{max_z:+.2f}σ",                  f"divario massimo (Francia · Δcammino)"),
    (_team_it_map.get(most_fragile, most_fragile),    "squadra più fragile"),
    (_team_it_map.get(least_fragile, least_fragile),  "squadra più robusta"),
])

# The 4×2 grid
for i, team in enumerate(TEAMS):
    for j, metric in enumerate(["avg_path_length", "efficiency"]):
        ax = fig.add_subplot(gs[i + 1, j])
        sub = df[(df.team == team) & (df.metric == metric)]
        rand_vals = sub[sub["mode"] == "random"]["damage"].dropna().values
        targ_val  = sub[sub["mode"] == "targeted"]["damage"].iloc[0]
        # Histogram of random null
        ax.hist(rand_vals, bins=22, color=TEAM_COLORS[team], alpha=0.55,
                edgecolor="white", linewidth=0.5)
        ax.axvline(0, color=MUTED, lw=1.0, zorder=1)
        # Random mean
        ax.axvline(rand_vals.mean(), color=INK_SOFT, linestyle=":",
                   linewidth=1.4, label="media casuale")
        # Targeted: three-pass glow effect for emphasis
        ax.axvline(targ_val, color=HIGHLIGHT, linewidth=6, alpha=0.18, zorder=2)
        ax.axvline(targ_val, color=HIGHLIGHT, linewidth=2.5, zorder=3,
                   label="mirata")
        # z and p chip in the corner
        z = (targ_val - rand_vals.mean()) / rand_vals.std(ddof=1)
        p_val = float(np.mean(rand_vals >= targ_val))
        sig = "***" if p_val < 0.001 else ("**" if p_val < 0.01 else ("*" if p_val < 0.05 else ""))
        ax.text(0.97, 0.92, f"z = {z:+.2f}  {sig}\np = {p_val:.3f}",
                transform=ax.transAxes, ha="right", va="top",
                fontsize=TEXT["annotation"], family="monospace",
                bbox=dict(boxstyle="round,pad=0.30", fc="white", ec=MUTED,
                          lw=0.8, alpha=0.96))
        # Title (panel-style)
        nice_metric = {"avg_path_length": "Δ lunghezza media cammini",
                       "efficiency": "Δ efficienza"}[metric]
        team_it = {"Argentina":"Argentina","France":"Francia","Croatia":"Croazia","Morocco":"Marocco"}[team]
        style_panel(ax, title=f"{team_it}   ·   {nice_metric}", subtitle=None)
        ax.set_xlabel("danno relativo" if i == 3 else "",
                      fontsize=TEXT["annotation"], color=INK_SOFT)
        ax.set_ylabel("conteggio" if j == 0 else "",
                      fontsize=TEXT["annotation"], color=INK_SOFT)
        if i == 0 and j == 0:
            ax.legend(loc="upper left", frameon=False,
                      fontsize=TEXT["annotation"])

plt.show()


**Cosa abbiamo imparato.**

- Per tutte e quattro le squadre, **togliere il pivot fa più danni della media delle rimozioni casuali**, su entrambe le metriche. Il giocatore top-1 in betweenness sta davvero facendo qualcosa che gli altri non fanno.
- L'**Argentina** è la squadra che soffre di più: $z = +4.66$ sulla Δlunghezza media dei cammini e $z = +3.51$ sulla Δefficienza. Otamendi è strutturalmente insostituibile — togliere lui produce un danno quattro volte la deviazione standard delle rimozioni casuali.
- La **Francia** è subito dietro su entrambe le metriche ($z = +4.76$ su Δcammino; $z = +3.11$ su Δefficienza). Togliere Tchouaméni è praticamente impossibile da replicare per caso.
- La **Croazia** è la più robusta delle quattro: $z = +1.43$ su Δcammino e $z = +1.75$ su Δefficienza. Togliere Gvardiol (o, in modo equivalente, Modrić — sono praticamente pari merito) sembra una perdita come tante altre. La squadra ha della ridondanza, coerente con un centrocampo di tre pivot quasi intercambiabili.
- Il **Marocco** sta in mezzo ($z \approx +1.7$ su entrambe le metriche): un pivot identificabile ma non così dominante come quelli di Argentina e Francia.


---
## 4. Togliere più giocatori — uno, due, tre

Togliere un solo giocatore ci dice quanto la squadra dipende da quel singolo nome. L'esperimento classico della Lezione 7 (Albert, Jeong & Barabási 2000) fa qualcosa di più: a ogni passo **ricalcoliamo la betweenness sul grafo rimasto** e togliamo il nuovo top-1. La curva del danno cumulativo ci dice se la squadra ha hub secondari intercambiabili oppure no.

Ci fermiamo dopo tre rimozioni: bastano per vedere le curve separarsi, ma non così tante da rendere la lunghezza media dei cammini priva di senso.


In [ ]:
prog_rows = []
for team in TEAMS:
    G_curr = graphs[team].copy()
    baseline = {m: f(G_curr) for m, f in metrics.items()}
    removed = []
    for step in range(4):
        post = {m: f(G_curr) for m, f in metrics.items()}
        prog_rows.append({
            "team":   team,
            "step":   step,
            "removed so far": ", ".join(short(n) for n in removed) or "—",
            "Δ avg path length": relative_damage(baseline["avg_path_length"],
                                                  post["avg_path_length"],
                                                  "avg_path_length"),
            "Δ efficiency":       relative_damage(baseline["efficiency"],
                                                  post["efficiency"],
                                                  "efficiency"),
        })
        if step < 3:
            btw = weighted_betweenness(G_curr)
            nxt = max(btw, key=btw.get)
            removed.append(nxt)
            G_curr.remove_node(nxt)

prog_df = pd.DataFrame(prog_rows)
print(prog_df.round(3).to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt
from adjustText import adjust_text

# =========================================================
# FIGURE
# =========================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=FIG["wide"],
    facecolor=BG
)

# =========================================================
# MAIN LOOP
# =========================================================

for ax, metric, ylabel, panel_title, panel_subtitle in zip(
    axes,
    ["Δ avg path length", "Δ efficiency"],
    ["Δ lunghezza media cammini (relativa)", "Δ efficienza (relativa)"],
    ["Danno cumulativo alla lunghezza dei cammini",
     "Danno cumulativo sull'efficienza"],
    ["di quanto la palla percorre più strada, in media",
     "quanta parte del flusso di rete viene persa"]
):

    texts = []

    # =====================================================
    # LINES
    # =====================================================

    for team in TEAMS:

        sub = prog_df[prog_df.team == team]
        color = TEAM_COLORS[team]

        # Glow
        ax.plot(
            sub.step,
            sub[metric],
            color=color,
            lw=6,
            alpha=0.18,
            zorder=1
        )

        # Main line
        ax.plot(
            sub.step,
            sub[metric],
            marker="o",
            markersize=9,
            linewidth=2.4,
            color=color,
            label={"Argentina":"Argentina","France":"Francia","Croatia":"Croazia","Morocco":"Marocco"}[team],
            zorder=3
        )

        # =================================================
        # STEP-1 LABEL
        # =================================================

        step1 = sub[sub.step == 1].iloc[0]
        pivot_name = step1["removed so far"]

        t = ax.text(
            1,
            step1[metric],
            f" −{pivot_name}",
            fontsize=TEXT["annotation"] - 0.5,
            color=color,
            fontweight="semibold",
            zorder=5
        )

        texts.append(t)

    # =====================================================
    # AUTO LABEL ADJUST
    # =====================================================

    adjust_text(
        texts,
        ax=ax,
        only_move={"text": "y"},
        arrowprops=dict(
            arrowstyle="-",
            color=INK_SOFT,
            lw=0.6,
            alpha=0.6
        )
    )

    # =====================================================
    # AXES
    # =====================================================

    ax.set_xticks([0, 1, 2, 3])

    ax.set_xticklabels([
        "baseline",
        "+ top-1",
        "+ top-2",
        "+ top-3"
    ])

    ax.set_ylabel(
        ylabel,
        fontsize=TEXT["annotation"],
        color=INK_SOFT
    )

    ax.set_xlabel(
        "rimozioni cumulative (betweenness ricalcolata)",
        fontsize=TEXT["annotation"],
        color=INK_SOFT
    )

    ax.legend(
        loc="upper left",
        frameon=False,
        fontsize=TEXT["annotation"]
    )

    ax.grid(
        True,
        alpha=0.25,
        axis="y"
    )

    # =====================================================
    # CLEAN PANEL TITLES
    # =====================================================

    ax.set_title(
    panel_title,
    fontsize=TEXT["annotation"] + 2,
    color=INK,
    pad=28,
    loc="left",
    fontweight="semibold"
)

    ax.text(
        0,
        1.01,
        panel_subtitle,
        transform=ax.transAxes,
        fontsize=TEXT["annotation"],
        color=INK_SOFT,
        ha="left"
    )

# =========================================================
# GLOBAL TITLE
# =========================================================

fig.suptitle(
    "Progressione dell'attacco — rimozione cumulativa dei top-3 per betweenness ricalcolata",
    fontsize=TEXT["fig_title"],
    fontweight="bold",
    color=INK,
    x=0.06,
    ha="left",
    y=0.98
)

# =========================================================
# GLOBAL SUBTITLE
# =========================================================

fig.text(
    0.06,
    0.92,
    "Le curve divergono nettamente al passo 1 (Francia in testa, Croazia in coda) e convergono parzialmente "
    "al passo 3 — la rete riesce ad assorbire una perdita mirata, non tre.",
    fontsize=TEXT["fig_subtitle"],
    color=INK_SOFT,
    ha="left"
)

# =========================================================
# FINAL LAYOUT
# =========================================================

plt.subplots_adjust(
    left=0.06,
    right=0.98,
    top=0.78,
    bottom=0.12,
    wspace=0.20
)

plt.show()

**Cosa abbiamo imparato.**
- Le curve di danno **crescono sempre, ma non in modo lineare**: il salto più grosso è fra il baseline e il passo 1, poi tendono a stabilizzarsi man mano che la ridondanza si esaurisce.
- L'**Argentina** ha il danno più grande sin dal *passo 1* (perdita di Otamendi: $\Delta$cammino = 0.30, $\Delta$efficienza = 0.21) e resta in testa fino al passo 3 (Otamendi → De Paul → Fernández): la curva di Δcammino sale fino a 0.69, la più alta delle quattro.
- La **Francia** segue subito dietro a ogni passo: Tchouaméni → Varane → Griezmann porta il danno cumulativo a $\Delta$cammino = 0.57.
- La **Croazia** è il caso opposto: danno minimo al passo 1, e anche al passo 3 resta la più contenuta — togliere Gvardiol, Modrić e Kovačić tutti insieme distrugge circa il 50% dell'integrità dei cammini. Avere tre hub quasi intercambiabili attutisce la perdita di uno, ma non quella di tutti e tre.
- È il risultato di Albert-Jeong-Barabási in versione mini: le reti con hub forti sono *robuste se perdono pochi nodi a caso, ma fragili se vengono attaccate proprio sugli hub*.


---
## 5. Mettendo tutto insieme — la RQ1 prevede la RQ3?

A questo punto abbiamo per ogni squadra due misure indipendenti:

- **Dominanza (dalla RQ1):** il rapporto top-1 / top-2 della weighted betweenness, calcolato nella §1.
- **Fragilità (dalla RQ3):** lo z-score medio sulle due metriche di danno della §3.

Se quello che abbiamo raccontato fin qui ha senso, le due misure devono essere **correlate positivamente**: più una squadra ha un pivot dominante, più dovrebbe soffrire quando glielo togliamo.

Con solo $n = 4$ squadre, ovviamente non stiamo facendo statistica vera e propria — stiamo facendo un **controllo di coerenza fra due esperimenti diversi sugli stessi dati**.


In [ ]:
# Build the joint table.
fragility = summary_df.groupby("team")["z"].mean().rename("fragility z").to_frame()
dom = pd.DataFrame({"team": TEAMS,
                    "top1/top2": setup_df.set_index("team").loc[TEAMS, "top1/top2"].values
                   }).set_index("team")
joint = dom.join(fragility)
print(joint.round(2).to_string())

r,  _ = pearsonr(joint["top1/top2"], joint["fragility z"])
rs, _ = spearmanr(joint["top1/top2"], joint["fragility z"])
print(f"\nPearson  r = {r:+.2f}")
print(f"Spearman ρ = {rs:+.2f}   (n = 4 teams)")


In [ ]:
# Flagship composite: scatter + four pitch mini-cards on the side.
# Style template: the "publication-ready" composite from notebook 04 §D.

fig = plt.figure(figsize=FIG["flagship"], facecolor=BG)
gs = fig.add_gridspec(
    nrows=4, ncols=12,
    left=0.05, right=0.97, top=0.86, bottom=0.10,
    hspace=1.0, wspace=0.6,
)

# --- Banner -----------------------------------------------------------------
fig.text(0.05, 0.965,
         "La dominanza del pivot predice la fragilità della rete",
         ha="left", va="top", fontsize=TEXT["fig_title"] + 2,
         fontweight="semibold", color=INK)
fig.text(0.05, 0.928,
         "Due esperimenti sulle stesse quattro squadre convergono sullo stesso ordine. "
         "Argentina la più fragile in assoluto, Francia la più dominante per dominanza "
         "del pivot, Croazia all'opposto — pivot intercambiabili e rete robusta.",
         ha="left", va="top", fontsize=TEXT["fig_subtitle"], color=INK_SOFT,
         wrap=True)

# --- Main panel: scatter ----------------------------------------------------
ax_main = fig.add_subplot(gs[0:4, 0:7])
# Best-fit line under data
slope, intercept = np.polyfit(joint["top1/top2"], joint["fragility z"], 1)
xs = np.linspace(joint["top1/top2"].min() * 0.92,
                 joint["top1/top2"].max() * 1.08, 50)
ax_main.plot(xs, slope * xs + intercept, "--", color=INK_SOFT, lw=1.4,
             alpha=0.6, zorder=1)
# Glow then sharp point per team
for team in TEAMS:
    x = joint.loc[team, "top1/top2"]
    y = joint.loc[team, "fragility z"]
    color = TEAM_COLORS[team]
    ax_main.scatter(x, y, s=2100, color=color, ec="white", lw=2.5,
                    alpha=0.18, zorder=2)
    ax_main.scatter(x, y, s=600, color=color, ec="white", lw=2.5, zorder=3)
    offsets = {"Argentina": (16, 14), "France": (-22, -22),
               "Croatia": (16, -14), "Morocco": (-12, 18)}
    ox, oy = offsets[team]
    team_it = {"Argentina":"Argentina","France":"Francia","Croatia":"Croazia","Morocco":"Marocco"}[team]
    ax_main.annotate(team_it, (x, y), xytext=(ox, oy), textcoords="offset points",
                     fontsize=13, fontweight="semibold", color=color)

# Correlation annotation in the corner
ax_main.text(0.97, 0.05,
             f"Pearson  r = {r:+.2f}\nSpearman ρ = {rs:+.2f}\n(n = 4)",
             transform=ax_main.transAxes, ha="right", va="bottom",
             fontsize=TEXT["annotation"], family="monospace",
             bbox=dict(boxstyle="round,pad=0.35", fc="white",
                       ec=MUTED, lw=0.8, alpha=0.96))

# Title for the panel: explicit, doesn't use style_panel (which collides with banner)
ax_main.set_title("Dominanza dalla RQ1   vs   fragilità dalla RQ3",
                  loc="left", pad=10,
                  fontsize=TEXT["panel_title"], fontweight="semibold", color=INK)
ax_main.set_xlabel("rapporto top-1 / top-2 della betweenness  (RQ1)",
                   fontsize=TEXT["annotation"] + 0.5, color=INK_SOFT)
ax_main.set_ylabel("z-score di fragilità  (RQ3)",
                   fontsize=TEXT["annotation"] + 0.5, color=INK_SOFT)
ax_main.grid(True, alpha=0.25)
# Limiti calcolati dai dati con un padding del 12%, così se i numeri
# cambiano in futuro nessun punto resta tagliato fuori (era successo
# con la Francia quando il top1/top2 = 3.56 cadeva oltre l'xlim 3.45).
_xlo, _xhi = joint["top1/top2"].min(), joint["top1/top2"].max()
_ylo, _yhi = joint["fragility z"].min(), joint["fragility z"].max()
_xpad = (_xhi - _xlo) * 0.12
_ypad = (_yhi - _ylo) * 0.12
ax_main.set_xlim(_xlo - _xpad, _xhi + _xpad)
ax_main.set_ylim(_ylo - _ypad, _yhi + _ypad)

# --- Side: four pitch mini-cards with the pivot highlighted -----------------
for i, team in enumerate(TEAMS):
    ax = fig.add_subplot(gs[i, 7:])
    G = graphs[team]
    btw = weighted_betweenness(G)
    pivot = max(btw, key=btw.get)
    color = TEAM_COLORS[team]
    # Pitch
    draw_pitch(ax, color=PITCH_BG, line=PITCH_LN, lw=0.6)
    # Edges in faint gold (compute wmax once outside the loop)
    wmax = max(d["weight"] for _, _, d in G.edges(data=True))
    for u, v, d in G.edges(data=True):
        if d["weight"] < 6: continue
        if "x" not in G.nodes[u] or "x" not in G.nodes[v]: continue
        x1, y1 = G.nodes[u]["x"], G.nodes[u]["y"]
        x2, y2 = G.nodes[v]["x"], G.nodes[v]["y"]
        ax.plot([x1, x2], [y1, y2], color="#bba266",
                alpha=0.18 + 0.6*(d["weight"]/wmax),
                lw=0.35 + 2.5*(d["weight"]/wmax),
                zorder=2)
    # Nodes — pivot in ACCENT (orange) to be distinguishable on any team colour
    btw_max = max(btw.values())
    for n in G.nodes():
        if "x" not in G.nodes[n]: continue
        x, y = G.nodes[n]["x"], G.nodes[n]["y"]
        size = 30 + 350 * btw.get(n, 0) / btw_max
        is_pivot = (n == pivot)
        node_color = ACCENT if is_pivot else color
        ec = "white"
        lw_e = 2.5 if is_pivot else 1.0
        zorder_n = 5 if is_pivot else 3
        # Glow under pivot
        if is_pivot:
            ax.scatter(x, y, s=size*2.4, color=ACCENT, alpha=0.30, zorder=4)
        ax.scatter(x, y, s=size, color=node_color, ec=ec, lw=lw_e,
                   zorder=zorder_n)
    # Title in white on the dark pitch
    team_it = {"Argentina":"Argentina","France":"Francia","Croatia":"Croazia","Morocco":"Marocco"}[team]
    ax.text(60, 75, f"{team_it}  ·  pivot = {short(pivot)}",
            ha="center", va="top",
            fontsize=TEXT["annotation"] + 0.5, color="white",
            fontweight="semibold",
            bbox=dict(boxstyle="round,pad=0.3", fc="#0d1f17", ec="none",
                      alpha=0.6))
    # Stats below
    z_score = float(fragility.loc[team, "fragility z"])
    dom_score = float(dom.loc[team, "top1/top2"])
    text_below_axes(
        ax,
        f"top1/top2 = {dom_score:.2f}     fragilità z = {z_score:+.2f}",
        y=-0.10, mono=True, box=True,
    )

# Caption line tying everything together
fig.text(0.05, 0.035,
         "I marker più grandi e luminosi identificano il pivot strutturale di ogni squadra. "
         "L'Argentina è l'unico outlier nettamente sopra la linea: pur avendo un divario top-1/top-2 "
         "moderato (1.84), perde in Otamendi un giocatore più critico di quanto il solo rapporto "
         "fra centralità lascia prevedere. La Francia, al contrario, è perfettamente predetta dal modello.",
         ha="left", va="top", fontsize=TEXT["annotation"], color=INK_SOFT,
         style="italic", wrap=True)

plt.show()


**Cosa abbiamo imparato.**
- **Pearson $r = +0.77$, Spearman $\rho = +0.80$**. Due esperimenti indipendenti su quattro squadre danno lo stesso ordine di fragilità: **Argentina ▸ Francia ▸ Marocco ▸ Croazia**.
- La retta predice molto bene tre dei quattro punti. La **Francia** (Tchouaméni) sta praticamente *sulla* linea — pivot estremamente dominante (top-1/top-2 = 3.56) e rete molto fragile ($z \approx 3.9$). La **Croazia** sta agli antipodi (top-1/top-2 = 1.04, $z \approx 1.6$) — pivot intercambiabili, rete robusta. Il **Marocco** è vicino alla Croazia, e ha senso: Hakimi e Amrabat hanno una betweenness quasi identica.
- L'**Argentina è il vero outlier sopra la linea**: ha uno z-score di fragilità ($\approx 4.5$) più alto di quanto il rapporto top-1 / top-2 (1.84) lascerebbe prevedere. Otamendi è più critico di quanto il solo divario fra centralità riesca a catturare. Una misura di dominanza più raffinata — ad esempio che tenga conto anche della *posizione spaziale* o del *reparto* del pivot rispetto agli altri top-betweenness — riporterebbe probabilmente l'Argentina sulla linea.


---
## In sintesi

- La **fragilità della rete dei passaggi** di una squadra si legge in larga parte dalla sua struttura: il divario fra top-1 e top-2 della weighted betweenness la predice bene (**$r = +0.77$, $\rho = +0.80$, $n = 4$**).
- $S(q)$ della NS07 non si può applicare a queste reti perché sono troppo dense per spezzarsi togliendo un nodo solo. Due alternative — Δ lunghezza media dei cammini e Δ efficienza — catturano lo stesso tipo di danno in un modo che funziona anche qui.
- Il risultato di Albert-Jeong-Barabási sulle reti scale-free vale *qualitativamente* anche su questi piccoli grafi pesati del calcio: l'attacco mirato fa molto più danno della rimozione casuale, e la differenza è maggiore quando un hub è molto dominante.
- **Lettura tattica.** L'Argentina è la squadra la cui rete dei passaggi dipendeva di più da un singolo giocatore (Otamendi). La Francia di Tchouaméni le sta subito dietro. I tre centrocampisti quasi intercambiabili della Croazia (Modrić, Brozović, Kovačić) erano l'opposto — un nucleo ridondante. L'Argentina ha vinto il torneo con una struttura più simile a quella francese che a quella croata, ed è una tensione interessante che già la RQ1 lascia intravedere.
- **Cosa rimane in sospeso.** L'Argentina sta sopra la linea di regressione, e questo suggerisce un miglioramento possibile: una misura di dominanza che tenga conto anche della *posizione spaziale* del pivot, non solo del divario fra le centralità. Lo segniamo come spunto per la sezione di future work del progetto.

---

### Cinque domande per testare se hai capito tutto

1. Perché $S(q)$ della NS07 non funziona su queste reti, e cosa usiamo al suo posto?
2. Cosa significa, per una squadra, avere uno z-score di fragilità alto pur con un rapporto top-1 / top-2 modesto? (Spunto: pensa all'Argentina.)
3. Perché le curve di attacco mirato delle quattro squadre tendono a convergere al passo 3?
4. Con $n = 4$, perché $r = +0.77$ *non* è una scoperta statistica? Cos'è invece?
5. Perché l'Argentina sta sopra la linea predittiva, e cosa cambierebbe se la misura di dominanza fosse anche spaziale?
